# RVC — Quy trình train đầy đủ: **giải thích + code từng bước**

Notebook này gộp **lý thuyết ngắn gọn** và **ô code có thể chạy** cho từng giai đoạn, tương ứng `training_pipeline/steps.py`.

**Cách dùng**

1. Mở workspace / kernel với **thư mục làm việc = `rvc_standalone`** (phải thấy `infer/`, `configs/`, `training_pipeline/`).
2. Chạy các ô **theo thứ tự từ trên xuống** (Bước A → B → 1 → 2 → 3 → 4 → tùy chọn).
3. Sửa biến trong **Bước B** cho khớp dataset của bạn.

**Mục lục:** Chuẩn bị → A khởi tạo → B tham số → 1 Preprocess → 2 F0 + Hubert → 3 Train → 4 Index FAISS → (tùy chọn) xuất model infer.

## Yêu cầu trước khi chạy

- Cài dependency: `pip install -r requirements.txt` (trong `rvc_standalone`).
- Tải trọng số: `python tools/download_assets.py` (Hubert, pretrained G/D, RMVPE, …).
- **`logs/mute/`** đủ thư mục mẫu im lặng — `filelist` khi train luôn thêm vài dòng mute. Nếu thiếu, Bước B sẽ báo.
- **FFmpeg** trên PATH (bước preprocess/đọc audio).

## Chuẩn bị — Thư mục chứa `.wav` của giọng cần học

Đây là **giọng đích** (timbre): sau này infer sẽ cố làm audio **giống giọng này**.

- Đặt nhiều file **`.wav`** (nói/hát, ít nền) vào một thư mục con, ví dụ `datasets/ten_giong/`.
- Thời lượng: thực tế nên **nhiều phút trở lên** (càng sạch càng tốt).

Ô code dưới **tạo thư mục**; bạn copy file audio vào bằng tay (hoặc đổi đường dẫn).

In [1]:
from pathlib import Path

thu_muc_audio = Path("datasets/giong_cua_toi")
thu_muc_audio.mkdir(parents=True, exist_ok=True)
print("Đặt file .wav vào:", thu_muc_audio.resolve())

Đặt file .wav vào: D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone\datasets\giong_cua_toi


---

## Bước A — Khởi tạo môi trường Python

**Làm gì:** đặt `cwd` về `rvc_standalone`, nạp `sys.path`, gọi `bootstrap()` để có `config` (Python, GPU, FP16, `device`, `preprocess_per`, …).

**Chạy:** một lần sau khi mở notebook hoặc **Restart kernel**.

**Ra sao với train:** các bước sau gọi subprocess `infer/modules/train/...` — `config.python_cmd` là interpreter, `config.device` dùng cho Hubert feature.

In [2]:
import logging
import os
import pathlib
import sys

STANDALONE_ROOT = pathlib.Path.cwd().resolve()
if not (STANDALONE_ROOT / "infer" / "modules" / "train" / "train.py").is_file():
    raise SystemExit(
        "cwd phải là rvc_standalone (có infer/modules/train/train.py). "
        "File → Open Folder → chọn rvc_standalone."
    )

os.chdir(STANDALONE_ROOT)
if str(STANDALONE_ROOT) not in sys.path:
    sys.path.insert(0, str(STANDALONE_ROOT))

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")

from training_pipeline.setup_env import bootstrap
from training_pipeline.params import TrainingParams
from training_pipeline import steps as train_steps

root, config = bootstrap()
assert root == STANDALONE_ROOT

print("Gốc:", STANDALONE_ROOT)
print("python:", config.python_cmd)
print("device (Hubert/feature):", config.device)
print("is_half:", config.is_half)
print("=== Bước A xong ===")

INFO | Found GPU NVIDIA GeForce RTX 3050 Laptop GPU
INFO | Half-precision floating-point: True, device: cuda:0


Gốc: D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone
python: d:\miniconda_env\conda_envs\rvc-my-server\python.exe
device (Hubert/feature): cuda:0
is_half: True
=== Bước A xong ===


---

## Bước B — Tham số huấn luyện (`TrainingParams`)

| Trường | Ý nghĩa |
|--------|--------|
| `experiment_name` | Tên thư mục `logs/<tên>/` chứa toàn bộ output. |
| `trainset_dir` | Thư mục chứa `.wav` gốc (đường dẫn tương đối so với `rvc_standalone`). |
| `sample_rate_label` | `32k` \| `40k` \| `48k` — phải **khớp** pretrained trong `assets/pretrained_v2/`. |
| `version` | `v1` (feature 256) hoặc `v2` (feature **768**, khuyến nghị). |
| `if_f0` | `True`: train model **có** nhánh cao độ; sinh `2a_f0`, `2b-f0nsf`. |
| `f0_method` | Với pipeline hiện tại: khác **`rmvpe_gpu`** → dùng `extract_f0_print.py` (vd. `rmvpe`, `harvest`, `pm`). Để dùng **`extract_f0_rmvpe.py`** (multi-GPU) xem docstring `TrainingParams.gpus_for_rmvpe` và đặt `f0_method="rmvpe_gpu"`. |
| `num_processes` | Số worker CPU cho preprocess / extract F0 (in). |
| `gpu_devices_train` | ID GPU train G/D, vd. `"0"` hoặc `"0-1"`. |
| `total_epochs` | Số epoch huấn luyện G/D. |
| `save_every_epoch` | Lưu checkpoint mỗi N epoch. |
| `batch_size` | Batch — tăng nếu còn VRAM. |
| `skip_index` | `True`: bỏ bước 4 FAISS (infer vẫn được với `index_rate=0`). |
| `extract_infer_pth` | Nếu `True`, `run_all` sẽ gọi luôn bước xuất model nhỏ; trong notebook ta tách ô riêng.
| `speaker_id` | ID speaker trong filelist (thường `0` nếu một giọng). |

Ô code dưới: **tạo `p`**, đếm `.wav`, kiểm tra `logs/mute`.

In [3]:
from pathlib import Path

p = TrainingParams(
    experiment_name="giong_A",
    trainset_dir="datasets/giong_cua_toi",
    sample_rate_label="40k",
    version="v2",
    if_f0=True,
    speaker_id=0,
    num_processes=4,
    f0_method="rmvpe",
    gpus_for_rmvpe="0",
    gpu_devices_train="0",
    total_epochs=50,
    save_every_epoch=5,
    batch_size=4,
    skip_index=False,
    extract_infer_pth=False,
    infer_weight_name="giong_A_infer",
)

ts = Path(p.trainset_dir)
if not ts.is_dir():
    raise SystemExit(f"Chưa có thư mục {ts} — tạo và copy .wav vào.")
wavs = list(ts.glob("*.wav")) + list(ts.glob("*.WAV"))
print("Số file .wav:", len(wavs))
if not wavs:
    raise SystemExit("Thư mục trainset không có .wav")

mm = train_steps.check_mute_template(STANDALONE_ROOT)
print("logs/mute:", "THIEU — copy mute từ bản RVC đầy đủ" if mm else "OK", mm or "")
print("=== Bước B xong ===")

Số file .wav: 2
logs/mute: OK 
=== Bước B xong ===


---

## Bước 1 — Preprocess (`step_preprocess` → `preprocess.py`)

**Mục đích:** đọc từng `.wav` trong `trainset_dir`, lọc, **cắt theo Slicer** (bỏ im lặng dài), chia clip ~vài giây, chuẩn hoá biên độ.

**Output chính:**

- `logs/<exp>/0_gt_wavs/*.wav` — audio **đúng sample rate đã chọn** (vd. 40 kHz) dùng làm **ground truth** khi train.
- `logs/<exp>/1_16k_wavs/*.wav` — bản **16 kHz** cho **Hubert** ở bước sau.

**Log:** `logs/<exp>/preprocess.log`

Chạy ô code dưới (có thể vài phút tùy dung lượng).

In [4]:
train_steps.step_preprocess(STANDALONE_ROOT, config, p)
print("=== Bước 1 xong ===")

INFO | Execute: "d:\miniconda_env\conda_envs\rvc-my-server\python.exe" infer/modules/train/preprocess.py "datasets/giong_cua_toi" 40000 4 "D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A" False 3.0


datasets/giong_cua_toi 40000 4 D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A False 3.0
start preprocess
datasets/giong_cua_toi 40000 4 D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A False 3.0
datasets/giong_cua_toi 40000 4 D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A False 3.0
datasets/giong_cua_toi 40000 4 D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A False 3.0
datasets/giong_cua_toi 40000 4 D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A False 3.0
datasets/giong_cua_toi/voice_ngan.wav	-> Success
end preprocess
=== Bước 1 xong ===


---

## Bước 2 — Trích F0 (tuỳ) + **đặc trưng Hubert** (`step_extract_f0_and_features`)

**Nếu `if_f0=True`:**

- Chạy script F0 (vd. `extract_f0_print.py` với `f0_method=rmvpe`) → thư mục **`2a_f0/`**, **`2b-f0nsf/`** (file `.npy` khớp tên clip với `0_gt_wavs`).

**Luôn chạy:**

- `extract_feature_print.py`: mỗi file trong `1_16k_wavs` → vector trong **`3_feature768/`** (v2) hoặc **`3_feature256/`** (v1).

**Log:** `logs/<exp>/extract_f0_feature.log`

Bước này cần **`assets/hubert/hubert_base.pt`**; RMVPE cho F0 cần **`assets/rmvpe/rmvpe.pt`** nếu dùng phương án RMVPE tương ứng.

Chạy ô dưới (thường **lâu hơn** bước 1).

In [5]:
train_steps.step_extract_f0_and_features(STANDALONE_ROOT, config, p)
print("=== Bước 2 xong ===")

INFO | Execute: "d:\miniconda_env\conda_envs\rvc-my-server\python.exe" infer/modules/train/extract/extract_f0_print.py "D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A" 4 rmvpe


d:\miniconda_env\conda_envs\rvc-my-server\lib\site-packages\pyworld\__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
infer/modules/train/extract/extract_f0_print.py D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A 4 rmvpe
d:\miniconda_env\conda_envs\rvc-my-server\lib\site-packages\pyworld\__init__.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
todo-f0-23
f0ing,now-0,all-23,-D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voic

INFO | Execute: "d:\miniconda_env\conda_envs\rvc-my-server\python.exe" infer/modules/train/extract_feature_print.py cuda:0 1 0 0 "D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A" v2 True


d:\miniconda_env\conda_envs\rvc-my-server\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
infer/modules/train/extract_feature_print.py cuda:0 1 0 0 D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A v2 True
exp_dir: D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standalone/logs/giong_A
load model(s) from assets/hubert/hubert_base.pt
Traceback (most recent call last):
  File "D:\DUT_ITF\Semester_10th\do_an_tot_nghiep\example_training_voice\Retrieval-based-Voice-Conversion-WebUI\rvc_standal

RuntimeError: extract_feature_print.py thất bại (exit 1). Đọc output phía trên và file logs/giong_A/extract_f0_feature.log — thường gặp: Hubert hỏng/thiếu, fairseq/torch lỗi, hoặc VRAM không đủ khi đưa model lên GPU.

---

## Bước 3 — Train **Generator G** + **Discriminator D** (`step_train`)

**Trước khi train:** `step_train` gọi `_write_filelist` — ghép các đường dẫn wav \| feature \| [f0] \| **speaker_id** thành `logs/<exp>/filelist.txt`, thêm mẫu **mute**, xáo `config.json` nếu chưa có.

- **G:** học sinh âm mang **timbre giọng đích** từ (content Hubert + F0 + spk).
- **D:** học phân biệt thật/giả để **ép G** tốt hơn — **infer không cần D**.

**Checkpoint:** `logs/<exp>/G_*.pth`, `D_*.pth`. Theo dõi **`train.log`**.

Ô code dưới chạy **rất lâu** (GPU).

In [ ]:
train_steps.step_train(STANDALONE_ROOT, config, p)
print("=== Bước 3 xong ===")

---

## Bước 4 — **FAISS index** (retrieval cho infer)

**Mục đích:** gom mọi vector trong `3_feature*` → có thể **gấp KMeans** nếu quá lớn → train index **IVF+Flat** → ghi **`added_IVF...index`** (file dùng khi infer; khác với `trained_...` trung gian).

Nếu **`skip_index=True`** trong Bước B: ô code sẽ bỏ qua (infer vẫn chạy nếu `index_rate=0`).

Nếu `.env` có **`outside_index_root`**, index có thể được copy/symlink ra `assets/indices/`.

In [ ]:
if p.skip_index:
    print("skip_index=True — bỏ qua bước 4")
else:
    for line in train_steps.step_train_index(STANDALONE_ROOT, config, p):
        print(line)
    print("=== Bước 4 xong ===")

---

## Bước 5 (tuỳ chọn) — Xuất file `.pth` **nhẹ** cho Infer / notebook infer

`G_2333333.pth` (hoặc checkpoint khác) thường **nặng** và có metadata train. `extract_small_model` tạo bản **chỉ cần cho inference** trong `assets/weights/` (tên từ `infer_weight_name`).

Có thể đặt `p.g_checkpoint_for_extract` = đường dẫn đầy đủ tới một `G_*.pth` cụ thể; để rỗng thì code tự chọn file G mới nhất trong `logs/<exp>/`.

In [ ]:
p.infer_weight_name = getattr(p, "infer_weight_name", "giong_A_infer")
p.g_checkpoint_for_extract = getattr(p, "g_checkpoint_for_extract", "")
msg = train_steps.step_extract_small_weights(STANDALONE_ROOT, p)
print(msg)

---

## Phụ lục — Gọi **một lần** cả pipeline (khi đã hiểu từng bước)

```python
# train_steps.run_all(STANDALONE_ROOT, config, p)
```

`run_all` chạy: preprocess → F0+feature → train → index (nếu không skip) → (nếu `extract_infer_pth=True`) xuất model nhỏ.

Khi debug, nên chạy **từng ô** phía trên để biết bước nào lỗi.

---

Tài liệu chỉ đọc (sâu hơn, không bắt buộc): `RVC_training_giai_thich_chi_tiet.ipynb`.